The goal is to add stop data to the raw data files. It then will get processed by the cleaning data to give us a complete look at all the info we have.

In [76]:
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import re
import os

In [77]:
def stopId_extract(route: str) -> dict:
    filehandle = open(f'{route}_stoplist.html')
    link_pattern = re.compile(r'^stophistory')
    soup = BeautifulSoup(markup= filehandle, features ='html.parser')
    links = soup.find_all('a', href=link_pattern)

    stop_link_dict = {}
    for link in links:
        stop_link_dict[link.text] = link['href'].removeprefix(f'stophistory?s=ttc.{route}.')[:4]

    return stop_link_dict

We actually want the IDs as keys and stops as values, since we only know the IDs from the raw data.

In [78]:
stop_dictionary = {str(float(value)):key for key, value in stopId_extract('506').items()}
# All good except 6934, which is missing a right bracket.
if ')' not in stop_dictionary['6934.0']:
    stop_dictionary['6934.0'] += ')'
stop_dictionary['NaN'] = np.nan
stop_dict_keys = set(stop_dictionary.keys())

In [79]:
display(stop_dictionary)

{'3832.0': 'Main Street Station at Bay 7',
 '4563.0': 'Main St at Gerrard St East',
 '2212.0': 'Norwood Rd',
 '5128.0': 'Glenmount Park Rd',
 '9431.0': 'Golfview Ave',
 '7178.0': 'Woodbine Ave',
 '8256.0': 'Kingsmount Park Rd',
 '4909.0': 'Bowmore Rd',
 '1128.0': 'Gerrard St East at Beaton Ave',
 '8206.0': 'Gerrard St East  at Coxwell Ave',
 '2615.0': 'Coxwell Ave at Lower Gerrard St East',
 '5884.0': 'Ashdale Ave',
 '7252.0': 'Woodfield Rd',
 '1027.0': 'Greenwood Ave',
 '5053.0': 'Prust Ave',
 '8406.0': 'Leslie St',
 '8448.0': 'Jones Ave',
 '314.0': 'Marjory Ave',
 '2640.0': 'Pape Ave',
 '7427.0': 'Carlaw Ave',
 '1441.0': 'Logan Ave',
 '1737.0': 'De Grassi St',
 '7548.0': 'Broadview Ave',
 '1447.0': 'St Matthews Rd',
 '9645.0': 'River St',
 '6451.0': 'Sumach St',
 '6970.0': 'Sackville St',
 '6146.0': 'Parliament St',
 '4615.0': 'Parliament St at Carlton St',
 '477.0': 'Ontario St',
 '5792.0': 'Sherbourne St',
 '9404.0': 'Jarvis St',
 '7599.0': 'Church St',
 '8396.0': 'Yonge St - Colle

In [80]:
raw_data_path = '../data/raw_data/schedule_data'
csvs = [pd.read_csv(os.path.join(raw_data_path,x)) for x in os.listdir(raw_data_path) if x[-4:] == '.csv']
raw_df = pd.concat(csvs, ignore_index=True)

/var/folders/6d/x65kjw5d1vl7fw7pnbx51yb80000gn/T/ipykernel_20905/122957432.py:2: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  csvs = [pd.read_csv(os.path.join(raw_data_path,x)) for x in os.listdir(raw_data_path) if x[-4:] == '.csv']
/var/folders/6d/x65kjw5d1vl7fw7pnbx51yb80000gn/T/ipykernel_20905/122957432.py:2: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  csvs = [pd.read_csv(os.path.join(raw_data_path,x)) for x in os.listdir(raw_data_path) if x[-4:] == '.csv']
/var/folders/6d/x65kjw5d1vl7fw7pnbx51yb80000gn/T/ipykernel_20905/122957432.py:2: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  csvs = [pd.read_csv(os.path.join(raw_data_path,x)) for x in os.listdir(raw_data_path) if x[-4:] == '.csv']
/var/folders/6d/x65kjw5d1vl7fw7pnbx51yb80000gn/T/ipykernel_20905/122957432.py:2: DtypeWarning: Columns (9) have mixed types. Speci

In [81]:
raw_df['Vehicle'] = raw_df['Vehicle'].astype(str)
veh_id = set(raw_df['Vehicle'].unique())

In [82]:
print(veh_id.symmetric_difference(stop_dict_keys))
print(len(veh_id.intersection(stop_dict_keys)))

{'8498.0', '1662.0', '3279.0', '1553.0', '4592.0', '4641.0', '8442.0', '3118.0', '8522.0', '4480.0', '3233.0', '3446.0', '7235.0', '8431.0', '3499.0', '1013.0', '8478.0', '7457.0', '4631.0', '4444.0', '8459.0', '7247.0', '8408.0', '7044.0', '3719.0', '3301.0', '8713.0', '8541.0', '4498.0', '8810.0', '8612.0', '8930.0', '4507.0', '9228.0', '8309.0', '3529.0', '4582.0', '9234.0', '3136.0', '7309.0', '3570.0', '4629.0', '8617.0', '8145.0', '4418.0', '8581.0', '8425.0', '1392.0', '8313.0', '7030.0', '4654.0', '9236.0', '8701.0', '4579.0', '8720.0', '8852.0', '4640.0', '8951.0', '8922.0', '8538.0', '4636.0', '4460.0', '3190.0', '3158.0', '8681.0', '8388.0', '8606.0', '7095.0', '3160.0', '8862.0', '9211.0', '4622.0', '8802.0', '7062.0', '3451.0', '3187.0', '3444.0', '3475.0', '36.0', '4501.0', '3189.0', '7427.0', '9209.0', '1653.0', '8559.0', '8854.0', '8613.0', '8404.0', '6053.0', '3533.0', '8824.0', '1014.0', '7257.0', '3496.0', '3420.0', '4639.0', '4402.0', '8733.0', '7103.0', '7204.0', '

In [89]:
len(raw_df['Schedule'].apply(lambda x: x[-10:-4] if type(x) == str else x).unique())

721